# Imports & paths

In [1]:
import pandas as pd
from pathlib import Path

DATA_RAW = Path("../data/raw")
DATA_INTERIM = Path("../data/interim")

DATA_INTERIM.mkdir(parents=True, exist_ok=True)

# Load raw data

In [2]:
customers = pd.read_csv(DATA_RAW / "customers.csv")
transactions = pd.read_csv(DATA_RAW / "transactions.csv")

customers.head(), transactions.head()

(  customer_id signup_date  true_lifetime_days
 0      C00000  2025-08-22                 204
 1      C00001  2025-03-07                 365
 2      C00002  2025-08-18                  48
 3      C00003  2025-09-22                  84
 4      C00004  2025-05-28                 113,
   customer_id transaction_date  amount
 0      C00000       2025-09-10  195.78
 1      C00000       2025-09-12   50.87
 2      C00000       2025-10-01  133.25
 3      C00000       2025-10-16   37.44
 4      C00000       2025-10-18  101.95)

# Preprocessing data

In [3]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   customer_id         3000 non-null   object
 1   signup_date         3000 non-null   object
 2   true_lifetime_days  3000 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 70.4+ KB


In [4]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46704 entries, 0 to 46703
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       46704 non-null  object 
 1   transaction_date  46704 non-null  object 
 2   amount            46704 non-null  float64
dtypes: float64(1), object(2)
memory usage: 1.1+ MB


## Missing & duplicate check 

In [5]:
customers_dup = customers.duplicated().sum()
print(customers_dup)

0


In [6]:
transactions_dup = transactions.duplicated().sum()
print(transactions_dup)

0


In [7]:
customers.isna().sum()

customer_id           0
signup_date           0
true_lifetime_days    0
dtype: int64

In [8]:
transactions.isna().sum()

customer_id         0
transaction_date    0
amount              0
dtype: int64

- dataset gồm ~3,000 khách hàng và ~46,000 giao dịch, không có missing hay duplicate.

- **signup_date**, **transaction_date** đang là object -> convert to datetime

- **true_lifetime_days** không dùng làm feature

## Date-time check

In [9]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

In [10]:
print("First day:" ,transactions["transaction_date"].min())
print("Last day: " ,transactions["transaction_date"].max())

print("Days diff:" ,(
    transactions["transaction_date"].max()
    - transactions["transaction_date"].min()
).days)

First day: 2025-01-03 00:00:00
Last day:  2025-12-31 00:00:00
Days diff: 362


- Dataset có timeline trong năm 2025 -> Phù hợp cho: window 30/60/90 ngày

## Amount check

In [11]:
negative_amount = (transactions["amount"] <= 0).sum()
print(negative_amount)

0


# User's transaction check

In [12]:
customer_ids_all = set(customers["customer_id"].unique())
customer_ids_with_txn = set(transactions["customer_id"].unique())
customer_ids_no_txn = customer_ids_all - customer_ids_with_txn

print("Total users in customer csv: ", len(customer_ids_all))
print("Total users in transaction csv: ",len(customer_ids_with_txn))
print("Total users with no transactions: ",len(customer_ids_no_txn))

Total users in customer csv:  3000
Total users in transaction csv:  2892
Total users with no transactions:  108


In [13]:
customers_no_txn = customers[customers["customer_id"].isin(customer_ids_no_txn)].copy()
customers_no_txn.to_csv(DATA_INTERIM / "customers_no_txn.csv", index=False)

customers_no_txn.shape

(108, 3)

- Với nhóm khách hàng chưa từng giao dịch trong toàn bộ giai đoạn quan sát, ta sẽ tách riêng ra để phía business xử lý

# Export clean file

In [14]:
customers = customers[~customers["customer_id"].isin(customer_ids_no_txn)].copy()
transactions = transactions[transactions["customer_id"].isin(customer_ids_with_txn)].copy()

customers.shape, transactions.shape

((2892, 3), (46704, 3))

In [15]:
transactions = transactions.sort_values(
    ["customer_id", "transaction_date"]
)

In [16]:
customers.to_parquet(
    DATA_INTERIM / "customers_clean.parquet",
    index=False
)

transactions.to_parquet(
    DATA_INTERIM / "transactions_clean.parquet",
    index=False
)